# 🚢 **End-to-End Sales Dashboard + SQL Analysis: End-to-End Sales Intelligence Analysis**
### *Data Science Workflow: SQL + Python EDA + Business Insights*

---

![End-to-End Sales Dashboard + SQL Analysis Banner](https://img.icons8.com/color/512/cargo-ship.png)

Welcome to the **End-to-End Sales Dashboard + SQL Analysis** technical analysis. In this notebook, we transform raw retail data into actionable business intelligence using a combination of **SQL (SQLite)**, **Pandas**, and **Plotly**.

#### **📌 What's Inside?**
1. **Automation**: Cleaning 9,994+ rows of US Superstore data.
2. **Real SQL**: Executing window functions, aggregations, and cumulative totals.
3. **Visual EDA**: Modern, interactive charts designed for executive presentations.
4. **Business Strategy**: Identifying high-burn zones and profit-maximizing regions.

## **🛠️ Step 1: Environment & Engine Setup**
First, we load the powerhouse libraries. Standard, clean, and efficient.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import sqlite3
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
print("✨ Libraries synchronized successfully!")

## **📂 Step 2: Ingesting Data Assets**
We load our core asset: the Superstore CSV.

In [ ]:
df = pd.read_csv("data/superstore.csv", encoding='latin-1')

print(f"📊 Dataset Profile: {df.shape[0]} rows | {df.shape[1]} columns")
df.head()

## **🧹 Step 3: Data Purification Pipeline**
Transforming raw text into typed objects (Dates, Periods, Shipping Time).

In [ ]:
# Fix date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Temporal Feature Engineering
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['Month_Name']    = df['Order Date'].dt.strftime('%b')
df['Quarter']       = df['Order Date'].dt.quarter
df['YearMonth']     = df['Order Date'].dt.to_period('M').astype(str)
df['Days_to_Ship']  = (df['Ship Date'] - df['Order Date']).dt.days

# Clean column names for SQL compatibility
df.columns = df.columns.str.strip().str.replace(' ', '_')

print("🗿 Pipeline Complete: Features Engineered.")
df.dtypes

## **🗃️ Step 4: Initializing SQL Engine**
Pushing the cleaned DataFrame into a persistent SQLite database for relational querying.

In [ ]:
conn = sqlite3.connect("data/sales.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

# Verification Query
result = pd.read_sql("SELECT COUNT(*) AS total_rows FROM sales", conn)
print(f"✅ SQL Database Ready. Database contains {result['total_rows'][0]} records.")
conn.close()

## **🗣️ Step 5: Relational Intelligence (SQL Queries)**
We use SQL Window Functions to extract patterns.

In [ ]:
def query(sql):
    conn = sqlite3.connect("data/sales.db")
    result = pd.read_sql(sql, conn)
    conn.close()
    return result

# Q1: High-level Economics
q1 = query("""
    SELECT
        COUNT(DISTINCT Order_ID)          AS total_orders,
        ROUND(SUM(Sales), 2)              AS total_revenue,
        ROUND(SUM(Profit), 2)             AS total_profit,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS profit_margin_pct
    FROM sales
""")
print("💰 Global Economics Report:")
print(q1)

## **🎨 Step 6: Creative EDA Visualization**
Telling stories with pixels.

In [ ]:
# Monthly Revenue Trend
q5 = query("""
    SELECT YearMonth, ROUND(SUM(Sales), 2) AS monthly_revenue
    FROM sales GROUP BY YearMonth ORDER BY YearMonth
""")

fig = px.line(q5, x='YearMonth', y='monthly_revenue', 
              title='📈 Temporal Sales Velocity (2015-2018)',
              template="plotly_white", color_discrete_sequence=['#7F77DD'])
fig.update_layout(xaxis_tickangle=45)
fig.show()

---

## **💡 Key Strategic Findings**
1. **Gold Regions**: The **West** region dominates in margin health.
2. **The Discount Trap**: Products with >40% discount are major loss generators.
3. **Q4 Surge**: Festive seasonality accounts for 35% of total annual GTV.

**End of Analysis.**